In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

C:\Users\arpit\anaconda3\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
113,"""This Man's Navy"" is, as other comments have i...",positive
427,Interesting topic. Pathetic delivery - script ...,negative
897,This is the Columbo that got directed by Steve...,positive
429,My wife and I both thought this film a watered...,negative
843,Watching The Tenants has been a interesting ex...,negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
113,this man s navy is comment indicated rare well...,positive
427,interesting topic pathetic delivery script dir...,negative
897,columbo got directed steven spielberg early po...,positive
429,wife thought film watered down made for tv bbc...,negative
843,watching tenant interesting experience me firs...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    267
positive    233
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
113,this man s navy is comment indicated rare well...,1
427,interesting topic pathetic delivery script dir...,0
897,columbo got directed steven spielberg early po...,1
429,wife thought film watered down made for tv bbc...,0
843,watching tenant interesting experience me firs...,0


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [11]:
import os
from dotenv import load_dotenv
import mlflow

try:
    import dagshub
    _DAGSHUB_AVAILABLE = True
except Exception:
    dagshub = None
    _DAGSHUB_AVAILABLE = False

load_dotenv()

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")
if mlflow_uri:
    try:
        mlflow.set_tracking_uri(mlflow_uri)
    except Exception as e:
        print(f"Failed to set MLflow tracking URI: {e}")

if _DAGSHUB_AVAILABLE:
    repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
    repo_name = os.getenv("DAGSHUB_REPO_NAME")
    if repo_owner and repo_name:
        try:
            dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
        except Exception as e:
            print(f"dagshub.init failed: {e}. Proceeding without dagshub.")
    else:
        print("DAGsHub repo owner/name not set; skipping dagshub.init().")
else:
    print("dagshub not available; proceeding without remote tracking.")

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as arpits-code

Initialized MLflow to track repo "arpits-code/mlops-text-classification-pipeline"

Repository arpits-code/mlops-text-classification-pipeline initialized!

2026/05/22 12:19:00 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f02205349e534558a62ba3543d7cb69e', creation_time=1779432540787, experiment_id='1', last_update_time=1779432540787, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [12]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-22 12:19:01,607 - INFO - Starting MLflow run...


2026-05-22 12:19:02,458 - INFO - Logging preprocessing parameters...


2026-05-22 12:19:04,766 - INFO - Initializing Logistic Regression model...


2026-05-22 12:19:04,767 - INFO - Fitting the model...


2026-05-22 12:19:04,785 - INFO - Model training complete.


...

2026-05-22 12:19:22,332 - INFO - Accuracy: 0.616


2026-05-22 12:19:22,334 - INFO - Precision: 0.6428571428571429


2026-05-22 12:19:22,334 - INFO - Recall: 0.5625


2026-05-22 12:19:22,335 - INFO - F1 Score: 0.6


🏃 View run suave-stag-924 at: https://dagshub.com/arpits-code/mlops-text-classification-pipeline.mlflow/#/experiments/1/runs/2eab304c10064f3c9bcc2efd80bac78e
🧪 View experiment at: https://dagshub.com/arpits-code/mlops-text-classification-pipeline.mlflow/#/experiments/1
